We are going to implement a character level transformer based model. The main core of this excercise is to understand the nitty gritties of the transformers architecture.

In [2]:
import torch

In [3]:
# !wget https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt

--2026-05-16 19:08:37--  https://raw.githubusercontent.com/karpathy/ng-video-lecture/refs/heads/master/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8002::154, 2606:50c0:8000::154, 2606:50c0:8001::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8002::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  2.07MB/s    in 0.5s    

2026-05-16 19:08:38 (2.07 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("Length of the dataset in characters: ", len(text))

Length of the dataset in characters:  1115394


In [6]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
# getting all the unique characters in the dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
# creating a mapping for the characters to integers (basically sort of a tokenizer for the character level model)
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[ch] for ch in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("Hii there"))
print(decode(encode("Hii there")))

[20, 47, 47, 1, 58, 46, 43, 56, 43]
Hii there


In [9]:
# encoding the entire dataset
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [10]:
# splitting up the dataset
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
len(train_data), len(val_data)

(1003854, 111540)

In [11]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"When context is {context}, target is: {target}")

When context is tensor([18]), target is: 47
When context is tensor([18, 47]), target is: 56
When context is tensor([18, 47, 56]), target is: 57
When context is tensor([18, 47, 56, 57]), target is: 58
When context is tensor([18, 47, 56, 57, 58]), target is: 1
When context is tensor([18, 47, 56, 57, 58,  1]), target is: 15
When context is tensor([18, 47, 56, 57, 58,  1, 15]), target is: 47
When context is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is: 58


In [12]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('Inputs:')
print(xb.shape)
print(xb)
print('Targets:')
print(yb.shape)
print(yb)

print('-----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f'When input is {context.tolist()}, the target is: {target}')

Inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
-----
When input is [24], the target is: 43
When input is [24, 43], the target is: 58
When input is [24, 43, 58], the target is: 5
When input is [24, 43, 58, 5], the target is: 57
When input is [24, 43, 58, 5, 57], the target is: 1
When input is [24, 43, 58, 5, 57, 1], the target is: 46
When input is [24, 43, 58, 5, 57, 1, 46], the target is: 43
When input is [24, 43, 58, 5, 57, 1, 46, 43], the target is: 39
When input is [44], the target is: 53
When input is [44, 53], the target is: 56
When input is [44, 53, 56], the target is: 1
When input is [44, 53, 56, 1], the target is: 58
When input is [44, 53

In [13]:
# bigram language model
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets) # we cannot call like this. This expects Channel (or C) dimension before. So, we have to reshape the logits
        
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        # get the predictions
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            # focus only on the last time step (cuz bigram model)
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        
        return idx

    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
# tensor(4.8786, grad_fn=<NllLossBackward0>) => loss
# loss should be around -ln(1/65) as it is pretty random right now so each character has the equal probability

# let's generate
idx = torch.zeros((1, 1), dtype=torch.long)
gen = m.generate(idx, max_new_tokens=100)
decode(gen[0].tolist())

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


'\nSKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp\nwnYWmnxKWWev-tDqXErVKLgJ'

In [14]:
# this is just for me to kind of get an intuition of multidimension splicing in tensors
temp = [[[13, 14],[15, 16], [17, 18]],[[19, 20],[20, 21],[22, 23]],[[24, 25],[26, 27],[28, 29]],[[30, 31],[32, 33],[34, 35]]]
temp = torch.tensor(temp)
temp[-1, :, :]

tensor([[30, 31],
        [32, 33],
        [34, 35]])

In [15]:
# creating a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [22]:
batch_size = 32

for steps in range(10000):
    # sample from batch
    xb, yb = get_batch('train')
    
    # forward pass
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    
    # backward pass
    loss.backward()
    
    # weight update
    optimizer.step()

print(loss.item())
    

2.450434446334839


In [26]:
# let's generate
idx = torch.zeros((1, 1), dtype=torch.long)
gen = m.generate(idx, max_new_tokens=500)
print(decode(gen[0].tolist()))

# now this is the simplest model output. It is obviously better than the random that we got above but still not great since the tokens are not talking to each other and we are just
# making predictions based on the last character


ARETwe atino g;
Pomarth thapes n allly s;
ARIARYof tol-Hesongshif: buinenod r d bl, I myondstheat
Surs herishosigre Whond d y w is th s ans ng
Somaseathest hy nor t ty or,
Pxre wat vemesthemarongy th n:
Wr cate 'sshand astt t:
G sanond s, Wharit hendolcle nd de yosteled itis w s.

TRI baror.
SABy by
LO 'd; m therdse.
ORCHelouror:
Rilld ss atidrrs urenathty heranthantoues e ch cthelofie,
BORouke tooou ck,
I t foubu borepa quthousoognereehof wour t wie:

QUMARI t yoway,
Anspett w IZVEThey th.
KINR


Mathematical trick in self-attention\
So, basically we want the previous words (or characters in our case) to predict the next word.\
We just want the previous words as the context.

In [31]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 
x = torch.randn(B, T, C)  # batch, time, channel
x.shape
# each input in the batch has 8 tokens. Currently, they are not talking to each other but we want them to talk for better understanding
# we want them to be talking to the previous tokens
# right now as a very naive approach, we can take the average of all the previous vectors and current one to kind of guess what is the current token w.r.t. the history context

torch.Size([4, 8, 2])

In [45]:
# we want to take average of all the previous token i.e. x[b, t] = mean(X[b, i]) where i <= t
xbow = torch.zeros((B, T, C)) # bow means bag of words here. Just kind of represent when we are taking frequency of the words or averaging them
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # shape would (t, C)
        xbow[b, t] = torch.mean(xprev, 0)

# very inefficient way to do this but this what we want to do right now

In [48]:
# to make it efficient, we can use matrix multiplication and we can use our understanding of matrix multiplication

torch.tril(torch.ones((3, 3))) # we get a lower triangle of 1 by this code. Since, we are averaging just the previous characters, we can use this concept

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [56]:
torch.manual_seed(43)
a = torch.tril(torch.ones((3, 3))) # 3x3
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(1, 10, (3, 2)).float() # 3x2
print("a==>")
print(a)
print("b==>")
print(b)
c = a @ b # 3x2
print("c==>")
print(c)
# so, in this way we are getting sum of just the previous previous row values (in our case, it would be time component)

a==>
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b==>
tensor([[5., 4.],
        [3., 5.],
        [5., 3.]])
c==>
tensor([[5.0000, 4.0000],
        [4.0000, 4.5000],
        [4.3333, 4.0000]])


In [66]:
# now we need to do this for our use case
wei = torch.tril(torch.ones(T, T)) # (T, T) because we want to take average of the time component (chars in our case) in the fashion mentioned above
wei = wei / wei.sum(1, keepdim=True)

xbow2 = wei @ x
xbow2.shape

torch.Size([4, 8, 2])

In [68]:
torch.allclose(xbow, xbow2)

True

In [75]:
# we can achieve the same thing with softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, 1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [89]:
# version 4: Self-attention
torch.manual_seed(1332)
B, T, C = 4, 8, 32

x = torch.randn(B, T, C)
# self attention is the answer to this problem that we do check the ineractions between the characters or words
# the way it does it that each node or token will give two vectors called Query (Q) and Key (K)
# Q represents what this particular token is looking for
# K represents the information that it can give or what does it contain
# so basically my (or that particular token's vectors) Q dot products with the K's of all the past tokens to gain the information of what can I get from all these tokens

# single head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # B, T, head_size
q = key(x) # B, T, head_size
# no communication has happened till now. We will do that now by using dot product
# so, all the query vectors are having their dot product with the key vectors
wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16. T) => (B, T, T)

tril = torch.tril(torch.ones((T, T)))
# wei = torch.zeros((T, T)) # we don't want the wei tensor to be uniform like in this case. We want it to be data dependent. As in what characters are interacting more and stuff.
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, -1)

v = value(x)
# instead of aggregating the raw values, we aggregate through v
out = wei @ v
# out = wei @ x

out.shape


torch.Size([4, 8, 16])

In [93]:
tr = torch.tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])
tr[-6:]

tensor([4, 5, 6, 7, 8, 9])